# LOFAR (Core / International stations) dynamic spectrum

This notebook plots dynamic spectra produced from LOFAR observations using
core or international stations. The data are stored as FITS files (one per
sub-band block) accompanied by a JSON metadata file holding the time and
frequency coverage. The FITS payload contains:

- `HDU[0].data`: dynamic spectrum, shape (n_freq, n_time)
- `HDU[1].data['FREQ']`: frequency axis in MHz
- `HDU[2].data['TIME']`: time axis as datetime

This is intended for LBA (Low Band Antenna) and HBA (High Band Antenna)
spectra; switch `band` below to choose which family of files to load.

**Workflow:** glob FITS files for the chosen station/date/band &rarr;
concatenate along the time axis &rarr; build a (time, freq) DataFrame &rarr;
remove the per-channel median background &rarr; plot.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

import json
from astropy.io import fits


## Configuration


In [ ]:
mydate = '2019-04-03'
year, month, day = mydate.split('-')

# LOFAR observation SAS ID and antenna band ('LBA' or 'HBA').
SASID = 'L700169'
band  = 'LBA'

data_dir = './sample_data/lofar'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


def load_lofar_block(fits_path):
    """
    Load one LOFAR dynamic-spectrum FITS block as a (time, freq) DataFrame.

    Parameters
    ----------
    fits_path : str
        Path to a `LOFAR_<YYYYMMDD>_*.fits` file.

    Returns
    -------
    pandas.DataFrame
        Index is the time axis, columns are the frequency axis (MHz).
    """
    with fits.open(fits_path) as hdul:
        spec  = np.asarray(hdul[0].data)                 # (freq, time)
        freqs = np.asarray(hdul[1].data['FREQ'])         # MHz
        times = pd.to_datetime(hdul[2].data['TIME'])     # datetime axis

    return pd.DataFrame(spec.T, index=times, columns=freqs)


## Load all FITS blocks for this station/date/band


In [ ]:
# Directory layout: <data_dir>/<SASID>_<YYYYMMDD>_<band>/LOFAR_<YYYYMMDD>_*.fits
block_dir = f'{data_dir}/{SASID}_{year}{month}{day}_{band}'

fits_files = sorted(glob.glob(f'{block_dir}/LOFAR_{year}{month}{day}_*.fits'))
# Some directories include an `..._<BAND>_OUTER.fits` summary file; drop it.
fits_files = [f for f in fits_files if not f.endswith(f'{band}_OUTER.fits')]

print(*fits_files, sep='\n')


In [ ]:
blocks = [load_lofar_block(f) for f in fits_files]
df_lofar = pd.concat(blocks, axis=0).sort_index()

# Drop any duplicate timestamps from block overlap.
df_lofar = df_lofar.loc[~df_lofar.index.duplicated(keep='first')]

print(f'Combined shape (time, freq): {df_lofar.shape}')
print(f'Time range: {df_lofar.index[0]} -> {df_lofar.index[-1]}')
print(f'Freq range: {df_lofar.columns.min():.2f} - {df_lofar.columns.max():.2f} MHz')


## Remove the per-channel median background


In [ ]:
df_lofar_nobkg = subtract_background_median(df_lofar)


## Plot the dynamic spectrum


In [ ]:
vmin = np.nanpercentile(df_lofar_nobkg.values, 5)
vmax = np.nanpercentile(df_lofar_nobkg.values, 99)

fig = plt.figure(figsize=(11, 6))
ax  = fig.add_subplot(111)
pc  = ax.pcolormesh(
    df_lofar_nobkg.index, df_lofar_nobkg.columns, df_lofar_nobkg.values.T,
    vmin=vmin, vmax=vmax, cmap='Spectral_r',
)
fig.colorbar(pc, ax=ax, pad=0.02, label='Intensity (a.u., background subtracted)')

ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'LOFAR  -  {SASID}  -  {band}  -  {mydate}')

fig.tight_layout()
fig.savefig(f'{outputs}/lofar_dyspec_{SASID}_{band}_{mydate}.png',
            bbox_inches='tight')
plt.show()
